In [0]:


df_orders = spark.sql(f"""
  SELECT * FROM workspace.olist_landing.bronze_orders
""")
display(df_orders)
df_orders.columns


In [0]:

df_orders.printSchema()

In [0]:
from pyspark.sql.functions import count

(
    df_orders
    .groupBy("order_id")
    .count()
    .filter("count > 1")
    .display()
)      


In [0]:
from pyspark.sql.functions import col, sum

df_orders.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in df_orders.columns
]).display()

df_orders.groupBy("order_status").count().display()

(
    df_orders
    .filter(col("order_delivered_customer_date").isNull())
    .groupBy("order_status")
    .count()
    .display()
)

In [0]:
%sql
SELECT  order_id, order_status,customer_id, order_delivered_carrier_date, order_delivered_customer_date
FROM workspace.olist_landing.bronze_orders WHERE order_status = 'delivered'
and order_delivered_customer_date is null

In [0]:
from pyspark.sql.functions import when, col

df_orders  =df_orders.withColumn("delivery_date_missing_flag", 
                                 when(
                                     (col("order_status") == "delivered") &
                                     (col("order_delivered_customer_date").isNull()),
                                     True
                                     ).otherwise(False)
                                )

df_orders.display()                                       
                                       
                                       

In [0]:
df_orders.printSchema()

In [0]:
%sql
use catalog workspace;
use schema olist_landing


In [0]:
df_orders_validate_delivered= spark.sql(f"""
  SELECT * FROM df_orders
  where order_delivered_customer_date < order_purchase_timestamp
""")
display(df_orders_validate_date)

df_orders_validate_approved= spark.sql(f"""
  SELECT  order_id,
  order_status, 
  order_delivered_customer_date,
  order_approved_at,
  timestampdiff(hour, order_delivered_customer_date, order_approved_at) as differenece_hour
  FROM df_orders
  where order_delivered_customer_date < order_approved_at
""")
display(df_orders_validate_approved)

In [0]:
from pyspark.sql.functions import when, col, expr

df_orders = (df_orders
                    .withColumn("approval_after_delivery_flag", 
                                 when(
                                     (col("order_status") == "delivered") &
                                     (col("order_delivered_customer_date") < col("order_approved_at")),
                                     True
                                     ).otherwise(False)
                                )
                    .withColumn("diff_hours", 
                              when(
                                     (col("order_status") == "delivered") &
                                     (col("order_delivered_customer_date") < col("order_approved_at")),
                                     expr("timestampdiff(hour, order_delivered_customer_date, order_approved_at)")
                                     ).otherwise(0)
                                )           

)

df_orders.display()          

In [0]:

df_orders.createOrReplaceTempView("df_orders")

spark.sql(f"""

  SELECT * FROM df_orders
  where diff_hours <> 0

""").display()

In [0]:
df_customers = spark.table("workspace.olist_landing.bronze_customers")
df_customers.createOrReplaceTempView("df_customers")

spark.sql(f"""

  SELECT customer_id, count(*)   
  FROM df_customers
  GROUP BY customer_id
  HAVING count(*)> 1
""").display()

In [0]:
df_orders_customer = (df_orders.alias("o")
                            .join(df_customers.alias("c"), col("o.customer_id") == col("c.customer_id"), "left")
                          
                    .select("o.*", "c.customer_unique_id", "c.customer_city", "c.customer_state", "c.customer_zip_code_prefix")
)
display(df_orders_customer)




In [0]:
print("before:", df_orders.count())
print("after:", df_orders_customer.count())


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.olist_silver
    

In [0]:
df_orders_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.olist_silver.orders_customer")

In [0]:
df_payments = spark.table("workspace.olist_landing.bronze_payments")
df_total_payment = df_payments.groupBy("order_id").agg(sum("payment_value").alias("total_payment"))
display(df_total_payment)
df_total_payment.write.format("delta").mode("overwrite").saveAsTable("workspace.olist_silver.orders_payment_totals")

In [0]:
df_customers = spark.read.table("workspace.olist_landing.bronze_customers")
#display(df_customers)
print("Total customer_id", df_customers.count())

print("Distinct customer_id_unique", df_customers.select("customer_unique_id").distinct().count())
print("Distinct customer_id", df_customers.select("customer_id").distinct().count())

df_customers.select("customer_unique_id","customer_id", "customer_zip_code_prefix", "customer_city","customer_state").write.format("delta").mode("overwrite").saveAsTable("workspace.olist_silver.customers")



In [0]:
%sql
select count(*) from workspace.olist_silver.customers